# <center> Лабораторна робота №12. Прогнозування затримок вильоту літаків з використанням різних алгоритмів бустінгу

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

In [2]:
from pathlib import Path
data_dir = Path('data')
if not (data_dir / 'flight_delays_train.csv').exists():
    data_dir = Path('lab12/data')
train = pd.read_csv(data_dir / 'flight_delays_train.csv')
test = pd.read_csv(data_dir / 'flight_delays_test.csv')
print('Розмір train:', train.shape)
print('Розмір test:', test.shape)


Розмір train: (100000, 9)
Розмір test: (100000, 8)


In [3]:
train.head()

,Month,DayofMonth,DayOfWeek,DepTime,UniqueCarrier,Origin,Dest,Distance,dep_delayed_15min
0,c-8,c-21,c-7,1934,AA,ATL,DFW,732,N
1,c-4,c-20,c-3,1548,US,PIT,MCO,834,N
2,c-9,c-2,c-5,1422,XE,RDU,CLE,416,N
3,c-11,c-25,c-6,1015,OO,DEN,MEM,872,N
4,c-10,c-7,c-6,1828,WN,MDW,OMA,423,Y


In [4]:
test.head()

,Month,DayofMonth,DayOfWeek,DepTime,UniqueCarrier,Origin,Dest,Distance
0,c-7,c-25,c-3,615,YV,MRY,PHX,598
1,c-4,c-17,c-2,739,WN,LAS,HOU,1235
2,c-12,c-2,c-7,651,MQ,GSP,ORD,577
3,c-3,c-25,c-7,1614,WN,BWI,MHT,377
4,c-6,c-6,c-3,1505,UA,ORD,STL,258


Отже, потрібно за часом вильоту літака, коду авіакомпанії-перевізника, місця вильоту та прильоту та відстанню між аеропортами вильоту та прильоту передбачити затримку вильоту більше 15 хвилин. Як найпростіший бенчмарк візьмемо логістичну регресію та дві ознаки, які найлегше взяти: `DepTime` та `Distance`. У такої моделі результат – 0.68202.

In [5]:
X_train, y_train = (
    train[["Distance", "DepTime"]].values,
    train["dep_delayed_15min"].map({"Y": 1, "N": 0}).values,
)
X_test = test[["Distance", "DepTime"]].values

X_train_part, X_valid, y_train_part, y_valid = train_test_split(
    X_train, y_train, test_size=0.3, random_state=17
)

In [6]:
logit = LogisticRegression(random_state=17)

logit.fit(X_train_part, y_train_part)
logit_valid_pred = logit.predict_proba(X_valid)[:, 1]

roc_auc_score(y_valid, logit_valid_pred)

np.float64(0.6795697123357751)

In [7]:
logit.fit(X_train, y_train)
logit_test_pred = logit.predict_proba(X_test)[:, 1]

results_dir = data_dir.parent / 'results'
results_dir.mkdir(exist_ok=True)
logit_path = results_dir / 'logit_2feat.csv'
pd.Series(logit_test_pred, name='dep_delayed_15min').to_csv(
    logit_path, index_label='id', header=True
)
print('Файл базової моделі створено:', logit_path)


Файл базової моделі створено: results/logit_2feat.csv


Побудувати покращену модель з використанням наступних підказок:
- ознаки `Distance` та `DepTime` брати без змін;
- створена ознака "маршрут" з вхідних ознак `Origin` та `Dest`;
- до ознак `Month`, `DayofMonth`, `DayOfWeek`, `UniqueCarrier` і "маршрут" застосувати OHE-перетворення (`LabelBinarizer`);
- видділити відкладену вибірку;
- навчати логістичну регресію і градієнтний бустінг (xgboost або sklearn.ensemble.GradientBoostingRegressor), гіперпараметри бустінгу налаштувати за результатами крос-валідації, спочатку ті, що відповідають за складність моделі, потім число дерев зафіксувати рівним 500 і налаштовувати крок градієнтного спуску;
- за допомогою `cross_val_predict` сформувати прогнози обох моделей на крос-валідації (саме передбачення ймовірності), налаштувати лінійну суміш відповідей логістичної регресії і градієнтного бустінгу вигляду $w_1 * p_{logit} + (1 - w_1) * p_{xgb}$, де $p_{logit}$ – прогноз логістичною регресією ймовірності класу 1, $p_{xgb}$ – аналогічно. Вага $w_1$ підбирається вручну. 
- як відповідь на тестовій вибірці брати аналогічну комбінацію відповідей двух моделей, але вже навчених на всій навчальній вибірці.

## Підготовка ознак

Створюємо ознаку `route` з аеропортів вильоту та прильоту. Категоріальні ознаки кодуємо `LabelBinarizer` у розрідженому форматі, щоб не створювати в пам’яті велику щільну матрицю.


In [8]:
from scipy.sparse import csr_matrix, hstack
from sklearn.preprocessing import LabelBinarizer, StandardScaler
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_predict

categorical_columns = ['Month', 'DayofMonth', 'DayOfWeek', 'UniqueCarrier', 'route']
numeric_columns = ['Distance', 'DepTime']

def add_route(frame):
    result = frame.copy()
    result['route'] = result['Origin'].astype(str) + '_' + result['Dest'].astype(str)
    return result

train_features = add_route(train)
test_features = add_route(test)
y = train['dep_delayed_15min'].map({'N': 0, 'Y': 1}).to_numpy()

numeric_scaler = StandardScaler()
train_numeric = csr_matrix(numeric_scaler.fit_transform(train_features[numeric_columns]))
test_numeric = csr_matrix(numeric_scaler.transform(test_features[numeric_columns]))
train_parts, test_parts = [train_numeric], [test_numeric]
encoders = {}
for column in categorical_columns:
    encoder = LabelBinarizer(sparse_output=True)
    encoder.fit(train_features[column].astype(str))
    encoders[column] = encoder
    train_parts.append(encoder.transform(train_features[column].astype(str)))
    test_parts.append(encoder.transform(test_features[column].astype(str)))

X = hstack(train_parts).tocsr()
X_test_full = hstack(test_parts).tocsr()
print('Кількість ознак після OHE:', X.shape[1])
print('Розмірність train:', X.shape, '| test:', X_test_full.shape)
print('Частка затриманих рейсів:', round(y.mean(), 4))


Кількість ознак після OHE: 4503
Розмірність train: (100000, 4503) | test: (100000, 4503)
Частка затриманих рейсів: 0.1904


## Логістична регресія та відкладена вибірка


In [9]:
X_train_part, X_valid, y_train_part, y_valid = train_test_split(
    X, y, test_size=0.3, random_state=17, stratify=y
)

logit = LogisticRegression(max_iter=2000, solver='saga', random_state=17, n_jobs=-1)
logit.fit(X_train_part, y_train_part)
logit_valid_prob = logit.predict_proba(X_valid)[:, 1]
print('ROC-AUC логістичної регресії на відкладеній вибірці:', round(roc_auc_score(y_valid, logit_valid_prob), 4))


ROC-AUC логістичної регресії на відкладеній вибірці: 0.7009


## Налаштування XGBoost

Спочатку підбираємо параметри складності дерева, далі фіксуємо 500 дерев і підбираємо `learning_rate`.


In [10]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=17)
base_xgb = XGBClassifier(
    objective='binary:logistic', eval_metric='auc', tree_method='hist',
    n_estimators=150, learning_rate=0.1, random_state=17, n_jobs=-1
)
complexity_params = {
    'max_depth': [3, 5],
    'min_child_weight': [1, 5],
    'subsample': [0.8]
}
complexity_search = GridSearchCV(
    base_xgb, complexity_params, scoring='roc_auc', cv=cv, n_jobs=1, verbose=1
)
complexity_search.fit(X_train_part, y_train_part)
print('Найкращі параметри складності:', complexity_search.best_params_)
print('Найкращий ROC-AUC на CV:', round(complexity_search.best_score_, 4))


Fitting 3 folds for each of 4 candidates, totalling 12 fits
Найкращі параметри складності: {'max_depth': 5, 'min_child_weight': 5, 'subsample': 0.8}
Найкращий ROC-AUC на CV: 0.7213


In [11]:
learning_rate_params = {'learning_rate': [0.03, 0.05, 0.1]}
tuned_xgb_base = XGBClassifier(
    objective='binary:logistic', eval_metric='auc', tree_method='hist',
    n_estimators=500, random_state=17, n_jobs=-1,
    **complexity_search.best_params_
)
learning_rate_search = GridSearchCV(
    tuned_xgb_base, learning_rate_params, scoring='roc_auc', cv=cv, n_jobs=1, verbose=1
)
learning_rate_search.fit(X_train_part, y_train_part)
best_xgb = learning_rate_search.best_estimator_
xgb_valid_prob = best_xgb.predict_proba(X_valid)[:, 1]
print('Оптимальний learning_rate:', learning_rate_search.best_params_['learning_rate'])
print('ROC-AUC XGBoost на CV:', round(learning_rate_search.best_score_, 4))
print('ROC-AUC XGBoost на відкладеній вибірці:', round(roc_auc_score(y_valid, xgb_valid_prob), 4))


Fitting 3 folds for each of 3 candidates, totalling 9 fits
Оптимальний learning_rate: 0.1
ROC-AUC XGBoost на CV: 0.7257
ROC-AUC XGBoost на відкладеній вибірці: 0.742


## Лінійна суміш прогнозів

Отримуємо ймовірності на крос-валідації для обох моделей та перевіряємо ваги від 0 до 1 з кроком 0.1.


In [12]:
cv_logit = LogisticRegression(max_iter=2000, solver='saga', random_state=17, n_jobs=-1)
cv_xgb = XGBClassifier(
    objective='binary:logistic', eval_metric='auc', tree_method='hist',
    n_estimators=500, random_state=17, n_jobs=-1,
    **complexity_search.best_params_,
    learning_rate=learning_rate_search.best_params_['learning_rate']
)
logit_cv_prob = cross_val_predict(cv_logit, X, y, cv=cv, method='predict_proba')[:, 1]
xgb_cv_prob = cross_val_predict(cv_xgb, X, y, cv=cv, method='predict_proba')[:, 1]

weights = np.arange(0, 1.01, 0.1)
blend_scores = [roc_auc_score(y, weight * logit_cv_prob + (1 - weight) * xgb_cv_prob) for weight in weights]
blend_table = pd.DataFrame({'Вага логістичної регресії': weights, 'ROC-AUC': blend_scores})
display(blend_table.round(4))
best_weight = float(weights[np.argmax(blend_scores)])
print('Найкраща вага логістичної регресії:', best_weight)
print('Найкращий ROC-AUC суміші:', round(max(blend_scores), 4))


,Вага логістичної регресії,ROC-AUC
0,0.0,0.7330
1,0.1,0.7340
2,0.2,0.7341
3,0.3,0.7333
4,0.4,0.7316
5,0.5,0.7289
6,0.6,0.7251
7,0.7,0.7199
8,0.8,0.7132
9,0.9,0.7047


Найкраща вага логістичної регресії: 0.2
Найкращий ROC-AUC суміші: 0.7341


## Прогноз для тестової вибірки


In [13]:
final_logit = LogisticRegression(max_iter=2000, solver='saga', random_state=17, n_jobs=-1).fit(X, y)
final_xgb = XGBClassifier(
    objective='binary:logistic', eval_metric='auc', tree_method='hist',
    n_estimators=500, random_state=17, n_jobs=-1,
    **complexity_search.best_params_,
    learning_rate=learning_rate_search.best_params_['learning_rate']
).fit(X, y)

final_probabilities = (
    best_weight * final_logit.predict_proba(X_test_full)[:, 1]
    + (1 - best_weight) * final_xgb.predict_proba(X_test_full)[:, 1]
)

# Один розгорнутий файл: характеристики рейсу, маршрут і прогноз затримки.
results_dir = data_dir.parent / 'results'
results_dir.mkdir(exist_ok=True)
detailed_predictions = test.copy()
detailed_predictions.insert(0, 'id', detailed_predictions.index)
detailed_predictions['route'] = detailed_predictions['Origin'] + ' → ' + detailed_predictions['Dest']
detailed_predictions['Ймовірність затримки'] = final_probabilities
detailed_predictions['Прогноз'] = np.where(
    final_probabilities >= 0.5,
    'Затримка понад 15 хвилин',
    'Без затримки понад 15 хвилин'
)

result_path = results_dir / 'flight_delays_predictions_detailed.csv'
detailed_predictions.to_csv(result_path, index=False)
print('Розгорнутий файл із прогнозами створено:', result_path)
print('Кількість рейсів із прогнозом затримки:', (final_probabilities >= 0.5).sum())
print('10 рейсів із найбільшою ймовірністю затримки:')
display(
    detailed_predictions.sort_values('Ймовірність затримки', ascending=False)
    [['id', 'route', 'UniqueCarrier', 'DepTime', 'Distance', 'Ймовірність затримки', 'Прогноз']]
    .head(10)
    .style.format({'Ймовірність затримки': '{:.2%}'})
)


Розгорнутий файл із прогнозами створено: results/flight_delays_predictions_detailed.csv
Кількість рейсів із прогнозом затримки: 2870
10 рейсів із найбільшою ймовірністю затримки:


,id,route,UniqueCarrier,DepTime,Distance,Ймовірність затримки,Прогноз
2741,2741,PHL → MHT,WN,2319,290,92.58%,Затримка понад 15 хвилин
12538,12538,LAS → SMF,WN,2310,397,90.36%,Затримка понад 15 хвилин
47084,47084,LAX → SMF,WN,2303,373,90.29%,Затримка понад 15 хвилин
15212,15212,BWI → FLL,WN,2300,925,89.16%,Затримка понад 15 хвилин
89789,89789,OAK → BUR,WN,2341,325,89.08%,Затримка понад 15 хвилин
67080,67080,LAX → SMF,WN,2347,373,88.13%,Затримка понад 15 хвилин
80669,80669,MCI → OKC,WN,2327,313,87.90%,Затримка понад 15 хвилин
27625,27625,BDL → TPA,WN,2314,1111,87.62%,Затримка понад 15 хвилин
61853,61853,MHT → PHL,WN,2319,290,87.47%,Затримка понад 15 хвилин
95562,95562,EWR → IAH,CO,2215,1400,87.26%,Затримка понад 15 хвилин


## Висновок

Порівняно логістичну регресію та XGBoost на однаковому наборі ознак. Логістична регресія є інтерпретованим базовим рішенням, а XGBoost враховує нелінійні залежності. Остаточну модель утворює зважена сума ймовірностей; найкраща вага обирається за максимальним ROC-AUC на крос-валідації.
